In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import InputLayer, Dense, Flatten, Dropout, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras import utils as np_utils
from sklearn.model_selection import StratifiedKFold
import kagglehub
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator


### Eu fiz varios testes e nao consegui chegar a uma valor de acuracia maior que 73%. Sei que nao e nem um pouco bom para um modelo, mas nao sei o que eu poderia fazr para melhorar. Evitei pesquisas na web, e fui tentando diferentes parametros.

In [2]:
x = np.load('/content/sample_data/cifar10_x (1).npy')
y = np.load('/content/sample_data/cifar10_y.npy')
# carrega  o dataset cifar10, esta em array numpy pq tive que reinicar sessao e o cifar 10 estava demorando 30 min para carregar

In [3]:
#x=tf.image.rgb_to_grayscale(x)
# transforma as imagens de rgb em tons de cinza

In [4]:
x = tf.image.resize(x, (32,32))
# redimensiona a imagem para 32x32
x = x.numpy()
# transformo em array numpy, pois o grayscaller transforma em objeto tensorflow

x= x.astype('float32')

In [5]:
x /= 255
# normaliza para estrar entre 0-1

In [6]:
y = np_utils.to_categorical(y,10)
# tansforma os dados de y em categorias, tipo o one hot encoder

In [7]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]])

In [8]:
x

array([[[[0.23137255, 0.24313726, 0.24705882],
         [0.16862746, 0.18039216, 0.1764706 ],
         [0.19607843, 0.1882353 , 0.16862746],
         ...,
         [0.61960787, 0.5176471 , 0.42352942],
         [0.59607846, 0.49019608, 0.4       ],
         [0.5803922 , 0.4862745 , 0.40392157]],

        [[0.0627451 , 0.07843138, 0.07843138],
         [0.        , 0.        , 0.        ],
         [0.07058824, 0.03137255, 0.        ],
         ...,
         [0.48235294, 0.34509805, 0.21568628],
         [0.46666667, 0.3254902 , 0.19607843],
         [0.47843137, 0.34117648, 0.22352941]],

        [[0.09803922, 0.09411765, 0.08235294],
         [0.0627451 , 0.02745098, 0.        ],
         [0.19215687, 0.10588235, 0.03137255],
         ...,
         [0.4627451 , 0.32941177, 0.19607843],
         [0.47058824, 0.32941177, 0.19607843],
         [0.42745098, 0.28627452, 0.16470589]],

        ...,

        [[0.8156863 , 0.6666667 , 0.3764706 ],
         [0.7882353 , 0.6       , 0.13333334]

In [9]:
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=33)
# define o numero de conjuntos de dados

In [14]:
rst = []
for i, t in kfold.split(x, np.zeros(shape=(y.shape[0],1))):
  # for que para treina o modelo para caa split do kfold, dividindo os dados de treino e uma matriz de target com 0
  model = Sequential()
  model.add(InputLayer(shape=(32,32,3)))
  # camada de entrada que recebe pixel para cada neuronio
  model.add(Conv2D(32, (3,3), activation='relu'))
  # operador de Covolução, vai fazer 32 filtros, matrizes de caracteristicas observando cantos, arrendondamento e etc.
  model.add(BatchNormalization()) # tentativa de melhorar
  model.add(MaxPooling2D(pool_size=(2,2)))
  # pegar os valores mais altos e colocar em uma matriz 3x3

  model.add(Conv2D(64, (3,3), activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPooling2D(pool_size=(2,2)))

  model.add(Flatten())
  # transforma a matriz em array
  model.add(Dense(units=256,activation='relu')) # camada densa
  model.add(Dropout(rate=0.5))

  model.add(Dense(units=256,activation='relu')) # camada densa
  model.add(Dropout(rate=0.5))

  model.add(Dense(units=10,activation='softmax'))
  # camada de saida com 10 neuronios 1 para cada classe
  model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
  # parametros para o modelo otimizar a rede
  model.fit(x[i],y[i], batch_size=256,epochs=10,validation_data=(x[t],y[t]))
  #para cada separação faz a o treino
  p = model.evaluate(x[t],y[t])
  # avalia a acuracia com dados que o modelo ainda nao viu
  rst.append(p[1])

Epoch 1/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 15s 41ms/step - accuracy: 0.3549 - loss: 1.8224 - val_accuracy: 0.2102 - val_loss: 3.4773
Epoch 2/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5163 - loss: 1.3633 - val_accuracy: 0.2963 - val_loss: 2.0457
Epoch 3/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5897 - loss: 1.1781 - val_accuracy: 0.5903 - val_loss: 1.1756
Epoch 4/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6339 - loss: 1.0571 - val_accuracy: 0.6105 - val_loss: 1.1606
Epoch 5/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6688 - loss: 0.9631 - val_accuracy: 0.6373 - val_loss: 1.0739
Epoch 6/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.6936 - loss: 0.8940 - val_accuracy: 0.6418 - val_loss: 1.0614
Epoch 7/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.7122 - loss: 0.8356 - val_accuracy: 0.6700 - val_loss: 0.9803
Epoch 8/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.7329 - loss: 0.7804 - val_acc

In [15]:
rst
# resultados de cada divisao

[0.702833354473114,
 0.7141666412353516,
 0.6909999847412109,
 0.7191666960716248,
 0.7139999866485596,
 0.7136666774749756,
 0.5985000133514404,
 0.690500020980835,
 0.7278333306312561,
 0.6961666941642761]

In [16]:
np.array(rst).mean()

np.float64(0.6967833399772644)

In [17]:
dtgen= ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
# cria imagens com diferentes parametros, para este dataset nao e necesssario mas segue um exemplo de como poderia ser feito

In [18]:
#train=dtgen.flow(x, batch_size = 128)
#model.fit(train, epochs=20, validation_data=y)